# 🌊 03 — Marine Health Index (MHI) Analysis
## Marine Biodiversity Ecosystem Assessment — Phase 5
**Shri Harsan M | M.Tech Data Science | SRM Institute**

This notebook covers:
- MHI component breakdown visualisation
- Biodiversity indices (Shannon H′, Simpson D, Pielou J′)
- Trophic structure analysis
- DTR (Detection Time Ratio) analysis
- Degradation robustness results

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import math

ROOT = Path('..').resolve()
MHI_JSON = ROOT / 'results' / 'biodiversity_health' / 'mhi_report.json'

# Load MHI report if available, else use known values
if MHI_JSON.exists():
    with open(MHI_JSON) as f:
        mhi_data = json.load(f)
    print('Loaded from mhi_report.json')
else:
    print('Using known Phase 5 test-set values')
    mhi_data = {
        'mhi': 70.59,
        'grade': 'GOOD',
        'alert': 'HEALTHY',
        'components': {
            'weighted_shannon_score':     90.9,
            'trophic_balance_score':      34.3,
            'apex_predator_score':        50.0,
            'indicator_presence_score':  100.0,
            'evenness_score':             90.9
        },
        'indices': {
            'shannon_H':     0.9986,
            'simpsons_D':    0.5955,
            'pielou_J':      0.9090,
            'species_richness': 3
        },
        'trophic': {
            'group_pcts':   {'Corallivores': 76.0, 'Herbivores': 24.0},
            'group_counts': {'Corallivores': 102,  'Herbivores': 31}
        },
        'counts': {'Butterflyfish': 27, 'Parrotfish': 31, 'Angelfish': 71}
    }

print(f"MHI Score: {mhi_data['mhi']:.2f}/100 — {mhi_data['grade']}")

## 1. MHI Component Breakdown

In [ ]:
components = mhi_data['components']
weights    = {'weighted_shannon_score': 0.30, 'trophic_balance_score': 0.25,
               'apex_predator_score': 0.20, 'indicator_presence_score': 0.15,
               'evenness_score': 0.10}
labels     = ["H'_weighted\n(×0.30)", "Trophic\nBalance\n(×0.25)",
               "Apex\nPredator\n(×0.20)", "Indicator\nPresence\n(×0.15)",
               "Pielou J'\n(×0.10)"]
keys       = list(weights.keys())
scores     = [components[k] for k in keys]
wts        = [weights[k]    for k in keys]
contrib    = [s*w for s, w in zip(scores, wts)]
colors     = ['#00e5cc', '#00ff9d', '#00c8ff', '#ffc200', '#8b5cf6']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'MHI Breakdown — Score: {mhi_data["mhi"]:.2f}/100 ({mhi_data["grade"]})',
             fontsize=13, fontweight='bold')

# Raw scores
bars = ax1.bar(labels, scores, color=colors, alpha=0.85, edgecolor='white')
ax1.set_ylim(0, 110); ax1.set_ylabel('Component Score (0–100)')
ax1.set_title('Raw Component Scores')
ax1.axhline(100, color='gray', linestyle=':', alpha=0.4)
for bar, val in zip(bars, scores):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.1f}',
             ha='center', fontweight='bold', fontsize=9)

# Weighted contributions
bars2 = ax2.bar(labels, contrib, color=colors, alpha=0.85, edgecolor='white')
ax2.set_ylabel('Weighted Contribution to MHI')
ax2.set_title(f'Weighted Contributions (sum = {sum(contrib):.2f})')
for bar, val in zip(bars2, contrib):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.2f}',
             ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('../results/biodiversity_health/mhi_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Biodiversity Indices Radar Chart

In [ ]:
idx = mhi_data['indices']

# Normalise all indices to 0-1
metrics = {
    "Shannon H'":     idx['shannon_H'] / math.log(3),  # max = ln(3) for 3 species
    "Simpson D":      idx['simpsons_D'],
    "Pielou J'":      idx['pielou_J'],
    "Richness":       idx['species_richness'] / 3,      # normalised to max 3
    "MHI/100":        mhi_data['mhi'] / 100,
}

labels_r = list(metrics.keys())
values   = list(metrics.values())
N        = len(labels_r)
angles   = [n / float(N) * 2 * math.pi for n in range(N)]
angles  += angles[:1]
values  += values[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.plot(angles, values, 'o-', linewidth=2, color='#00ff9d')
ax.fill(angles, values, alpha=0.2, color='#00ff9d')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_r, size=11)
ax.set_ylim(0, 1)
ax.set_title('Biodiversity Health Radar', size=13, fontweight='bold', pad=20)
ax.grid(color='gray', linestyle=':', alpha=0.5)

# Add value annotations
for angle, val, lbl in zip(angles[:-1], values[:-1], labels_r):
    ax.annotate(f'{val:.3f}', xy=(angle, val), xytext=(angle, val+0.07),
                ha='center', fontsize=9, color='#00c8ff', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/biodiversity_health/biodiversity_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. DTR Analysis

In [ ]:
# DTR grade bands
dtr_bands = [
    (0.0, 0.2, 'Excellent', '#00ff9d'),
    (0.2, 0.4, 'Good',      '#00e5cc'),
    (0.4, 0.6, 'Moderate',  '#ffc200'),
    (0.6, 0.8, 'Fair',      '#ff8c00'),
    (0.8, 1.0, 'Poor',      '#ff3d5a'),
]

fig, ax = plt.subplots(figsize=(12, 3))
for lo, hi, label, color in dtr_bands:
    ax.barh(0, hi-lo, left=lo, height=0.5, color=color, alpha=0.85)
    ax.text((lo+hi)/2, 0, label, ha='center', va='center', fontweight='bold', fontsize=10)

ax.set_xlim(0, 1)
ax.set_xlabel('DTR (Detection Time Ratio)  =  First Complete Frame / Total Frames')
ax.set_yticks([])
ax.set_title('DTR Grade Bands\n'
             'Formula: DTR = frame_of_first_complete_species_set / total_frames\n'
             'Time Score = (1 − DTR) × 100  |  Combined = 0.70×MHI + 0.30×Time_Score',
             fontsize=10)
ax.axvline(0.2, color='white', linewidth=2, linestyle='--')
ax.axvline(0.4, color='white', linewidth=2, linestyle='--')
ax.axvline(0.6, color='white', linewidth=2, linestyle='--')
ax.axvline(0.8, color='white', linewidth=2, linestyle='--')

plt.tight_layout()
plt.savefig('../results/biodiversity_health/dtr_bands.png', dpi=150, bbox_inches='tight')
plt.show()

print('DTR Formula Summary')
print('='*50)
print(f'  DTR = First_complete_frame / Total_frames')
print(f'  Time_Score = (1 - DTR) * 100')
print(f'  Combined   = 0.70 * MHI + 0.30 * Time_Score')
print()
print(f'  MHI (test set) = {mhi_data["mhi"]:.2f}')
print(f'  Example DTR=0.15 → Time_Score=85.0 → Combined={0.70*70.59 + 0.30*85.0:.2f}')